# 08 — Final QC manifest and publication results summary

Collects the primary, bootstrap, and complete-case results already written by stages 04/04c/05/06 into a single publication-facing summary, and produces a full file manifest (path/size/MD5) of the delivered repository for traceability (Methods 2.13).

## Environment setup

In [ ]:
import os, sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/<your-org>/<your-repo>.git"  # TODO: set to the published GitHub repo URL
REPO_DIR_NAME = "pcos_network_architecture_jcem_v3_jcem_10of10"


def _in_colab():
    try:
        import google.colab  # noqa: F401
        return True
    except ImportError:
        return False


if _in_colab():
    PROJECT_ROOT = Path("/content") / REPO_DIR_NAME
    if not PROJECT_ROOT.exists():
        subprocess.run(["git", "clone", REPO_URL, str(PROJECT_ROOT)], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                     "scikit-learn", "networkx", "statsmodels", "openpyxl"], check=True)
else:
    PROJECT_ROOT = Path.cwd().resolve()
    while not ((PROJECT_ROOT / "data_raw").exists() and (PROJECT_ROOT / "outputs").exists()):
        if PROJECT_ROOT.parent == PROJECT_ROOT:
            break
        PROJECT_ROOT = PROJECT_ROOT.parent

os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))
from scripts._pipeline_common import *  # noqa: F401,F403

print("Project root:", PROJECT_ROOT)


In [ ]:
import json
import pandas as pd

primary_summary = json.loads((PROJECT_ROOT / 'outputs' / '04_primary_graphical_lasso' / 'primary_graphical_lasso_model_summary.json').read_text())
metrics = pd.read_csv(PROJECT_ROOT / 'outputs' / '05_centrality_bridge' / 'Table_primary_centrality_bridge_metrics.csv')
boot = pd.read_csv(PROJECT_ROOT / 'outputs' / '06_bootstrap_stability' / 'centrality_bootstrap_summary_primary.csv')
cc_summary = json.loads((PROJECT_ROOT / 'outputs' / '04c_complete_case_network' / 'complete_case_model_summary.json').read_text())
cc_comparison = json.loads((PROJECT_ROOT / 'outputs' / '04c_complete_case_network' / 'complete_case_vs_primary_comparison.json').read_text())

top_bridge_primary = metrics.sort_values('bridge_strength', ascending=False).head(8)
top_central_primary = metrics.sort_values('strength', ascending=False).head(8)
bootstrap_top_bridge = boot.sort_values('bridge_strength_median', ascending=False).head(8)

pub = {
    'primary_summary': primary_summary,
    'top_bridge_primary': top_bridge_primary.to_dict('records'),
    'top_central_primary': top_central_primary.to_dict('records'),
    'bootstrap_top_bridge': bootstrap_top_bridge[[
        'feature', 'strength_median', 'strength_p025', 'strength_p975',
        'bridge_strength_median', 'bridge_p025', 'bridge_p975', 'domain',
    ]].to_dict('records'),
    'methodological_assessment': (
        'JCEM v3: direct-biomarker primary network, EBIC + density-constrained sparse selection, '
        '500-bootstrap stability, alpha/gamma sensitivity, derived variables only in secondary analyses. '
        'Complete-case sensitivity network (stage 04c, n=885) now implemented: 90% of primary edges preserved, '
        'bridge-strength Spearman rho=0.94, top-4 bridge biomarkers (SHBG, INS0, TG, HDL) identical rank order to primary.'
    ),
    'complete_case_sensitivity': {'summary': cc_summary, 'comparison_vs_primary': cc_comparison},
}

out_dir = PROJECT_ROOT / 'outputs' / '08_final_qc_manifest'
out_dir.mkdir(parents=True, exist_ok=True)
(out_dir / 'publication_results_summary_v3.json').write_text(json.dumps(pub, indent=2), encoding='utf-8')
print(json.dumps(pub['primary_summary'], indent=2))


### File manifest (path, size, MD5) of the delivered repository

In [ ]:
import hashlib

files = []
for f in sorted(PROJECT_ROOT.rglob('*')):
    if not f.is_file():
        continue
    if '__pycache__' in f.parts or '.git' in f.parts or '.ipynb_checkpoints' in f.parts:
        continue
    files.append({
        'path': str(f.relative_to(PROJECT_ROOT)).replace('\\', '/'),
        'size_bytes': f.stat().st_size,
        'md5': hashlib.md5(f.read_bytes()).hexdigest(),
    })
pd.DataFrame(files).to_csv(out_dir / 'file_manifest_v3.csv', index=False)
print(f'Files in manifest: {len(files)}')
